# Day 24: Integrating Docling for Complex PDF Parsing

Welcome to Day 24 of your AI Engineering journey! In Phase 2, we are establishing robust Vector Databases & RAG Foundations.

## Core Theory (Just-in-Time)

**Why Docling?**
In a production RAG system, your documents are rarely clean text files. They are often complex PDFs, Word documents, or PowerPoints containing embedded tables, images, multiple columns, and headers/footers.
Standard text extraction libraries (like `PyPDF2` or `pdfplumber`) often fail dramatically on these layouts. They read text line-by-line, which destroys the semantic structure of a multi-column page or a nested table. If you chunk and embed a broken table, the LLM will hallucinate when queried about that data.

**How Docling Works:**
`Docling` (by IBM) uses advanced layout analysis (often powered by vision models underneath) to semantically understand the document structure. It accurately reconstructs tables, headings, and paragraphs, and exports them into structured Markdown. Markdown is the optimal format for LLMs because it explicitly preserves hierarchical structure (e.g., `# Heading 1`, `| Table Header |`) while remaining extremely token-efficient.

**The Pipeline:**
1. Ingest complex PDF (with tables).
2. Process with `Docling` to output structured Markdown.
3. Chunk the Markdown using a Markdown-aware text splitter.
4. Vectorize and store in a Vector DB (like Qdrant) for retrieval.

## Code Implementation

Let's look at a production-grade example of parsing a complex document using Docling. We'll extract the content into Markdown and enforce strict type hinting and robust error handling.

In [1]:
import logging
from typing import Optional
from docling.document_converter import DocumentConverter
from pydantic import BaseModel, Field, ValidationError

# Setup production-grade logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ParsedDocument(BaseModel):
    """Schema for a successfully parsed document."""
    source_url: str = Field(..., description="The original URL or path of the document.")
    markdown_content: str = Field(..., description="The extracted markdown content.")

def parse_pdf_to_markdown(source_url: str) -> Optional[ParsedDocument]:
    """
    Parses a PDF document from a URL or local file into structured Markdown using Docling.
    
    Args:
        source_url (str): The path or URL to the document (e.g., PDF).
        
    Returns:
        Optional[ParsedDocument]: The validated document schema containing markdown, 
                                  or None if parsing failed.
    """
    logger.info(f"Initializing Docling DocumentConverter...")
    try:
        # In production, converter initialization can be cached or pooled if processing many docs.
        converter = DocumentConverter()
        
        logger.info(f"Converting document from: {source_url}")
        result = converter.convert(source_url)
        
        # Export the robust document model directly to Markdown
        md_text = result.document.export_to_markdown()
        
        # Validate with Pydantic
        parsed_doc = ParsedDocument(
            source_url=source_url,
            markdown_content=md_text
        )
        logger.info("Document successfully parsed and validated.")
        return parsed_doc
        
    except ValidationError as ve:
        logger.error(f"Data validation error for {source_url}: {ve}")
        return None
    except Exception as e:
        logger.error(f"Failed to parse document {source_url}. Error: {str(e)}")
        return None

# Example Usage
# We use a sample URL supported by Docling's typical test cases
SAMPLE_URL = "https://arxiv.org/pdf/2408.09869.pdf"  # Docling paper PDF as an example
parsed_result = parse_pdf_to_markdown(SAMPLE_URL)

if parsed_result:
    print("\n--- Extracted Markdown Snippet ---\n")
    print(parsed_result.markdown_content[:500] + "\n... [Truncated]")

2026-08-18 16:09:39,564 - INFO - Initializing Docling DocumentConverter...


2026-08-18 16:09:39,821 - INFO - Converting document from: https://arxiv.org/pdf/2408.09869.pdf


2026-08-18 16:09:40,147 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-08-18 16:09:40,292 - INFO - Going to convert document batch...


2026-08-18 16:09:40,294 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 1f82f44452ba377fa63e310478792c2e


...

[WARNING] 2026-08-18 16:10:42,196 [RapidOCR] main.py:132: The text detection result is empty


2026-08-18 16:10:42,197 - WARNING - RapidOCR returned empty result!


2026-08-18 16:11:45,763 - INFO - Finished converting document 2408.09869.pdf in 125.94 sec.


2026-08-18 16:11:45,905 - INFO - Document successfully parsed and validated.



--- Extracted Markdown Snippet ---

<!-- image -->

## Docling Technical Report

## Version 1.0

Christoph Auer Maksym Lysak Ahmed Nassar Michele Dolfi Nikolaos Livathinos Panos Vagenas Cesar Berrospi Ramis Matteo Omenetti Fabian Lindlbauer Kasper Dinkla Lokesh Mishra Yusik Kim Shubham Gupta Rafael Teixeira de Lima Valery Weber Lucas Morin Ingmar Meijer Viktor Kuropiatnyk Peter W. J. Staar

AI4K Group, IBM Research R¨ uschlikon, Switzerland

## Abstract

This technical report introduces Docling , an easy to use, self-contained, MI
... [Truncated]


## Common Pitfalls

1. **OCR Overhead vs. Native Text:** If a PDF has native digital text, basic parsers are fast. Docling applies deeper layout analysis, which can be computationally expensive (and slower) but yields far higher accuracy for tables and columns. In high-throughput systems, pre-filter documents to decide if heavy layout parsing is necessary.
2. **Token Limits and Table Chunking:** While Markdown tables are great, a massive table with 1000 rows will exceed context windows when chunked poorly. Even with Markdown, you must use intelligent splitting strategies (like LangChain's `MarkdownHeaderTextSplitter`) to keep chunks semantically cohesive.
3. **Hallucinating APIs:** Many developers assume LLMs can parse raw PDFs directly. While multimodal models can read image slices of a PDF, structured conversion to text (via Docling) remains vastly cheaper, faster, and more deterministically searchable in a Vector DB.

## Practical Lab / Homework

**Your Task:** 
1. Take the Markdown output generated above.
2. Use LangChain's `MarkdownHeaderTextSplitter` to split the text semantically based on headers.
3. Store the resulting chunks into a local Qdrant memory instance.
4. Query the points to verify successful insertion.

Below is the complete, working production-grade reference implementation for your lab.

In [2]:
import uuid
from typing import List
from langchain_text_splitters import MarkdownHeaderTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def process_and_index_markdown(markdown_text: str, collection_name: str = "docling_parsed_docs") -> None:
    """
    Splits markdown text via headers and indexes it into a local Qdrant collection using a dummy vector.
    
    Args:
        markdown_text (str): The markdown text to process.
        collection_name (str): The name of the Qdrant collection.
    """
    logger.info("Splitting markdown by headers...")
    # Define headers to split on
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    splits = markdown_splitter.split_text(markdown_text)
    
    logger.info(f"Generated {len(splits)} chunks.")
    if not splits:
        logger.warning("No splits generated. Text might be too short or lack headers.")
        return

    logger.info("Initializing in-memory Qdrant Client...")
    client = QdrantClient(":memory:")
    
    # Create collection
    # We use vector size 384 assuming a standard small embedding model like all-MiniLM-L6-v2
    # However, to avoid an external embedding dependency for this strict lab setup, 
    # we will use zeroed dummy vectors to demonstrate the architectural indexing flow.
    vector_size = 3
    client.recreate_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    
    logger.info("Preparing points for Qdrant...")
    points = []
    for i, split in enumerate(splits):
        # For demonstration without a live embedding API, use a dummy vector
        dummy_vector = [0.1, 0.2, 0.3]
        
        point = PointStruct(
            id=str(uuid.uuid4()),
            vector=dummy_vector,
            payload={
                "page_content": split.page_content,
                "metadata": split.metadata
            }
        )
        points.append(point)
        
    # Upsert points
    client.upsert(
        collection_name=collection_name,
        points=points
    )
    logger.info(f"Successfully indexed {len(points)} chunks into Qdrant collection '{collection_name}'.")
    
    # Query back to verify using the modern query_points API
    query_results = client.query_points(
        collection_name=collection_name,
        query=[0.1, 0.2, 0.3], # Dummy query matching the vector
        limit=1
    )
    
    print("\n--- Verification Query Result ---")
    if query_results.points:
        top_point = query_results.points[0]
        print(f"ID: {top_point.id}")
        print(f"Metadata: {top_point.payload.get('metadata', {})}")
        content = top_point.payload.get('page_content', '')
        print(f"Content snippet: {content[:100]}...")
    else:
        print("No results found.")

# Execute the lab integration
if parsed_result:
    process_and_index_markdown(parsed_result.markdown_content)
else:
    logger.error("Cannot run lab. Parsed document was None.")


2026-08-18 16:11:48,818 - INFO - Splitting markdown by headers...


2026-08-18 16:11:48,823 - INFO - Generated 22 chunks.


2026-08-18 16:11:48,824 - INFO - Initializing in-memory Qdrant Client...


/tmp/ipykernel_19425/3728344133.py:39: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
2026-08-18 16:11:48,843 - INFO - Preparing points for Qdrant...


2026-08-18 16:11:48,848 - INFO - Successfully indexed 22 chunks into Qdrant collection 'docling_parsed_docs'.



--- Verification Query Result ---
ID: 0e87137b-2595-4e0d-8f60-594bddd08bf8
Metadata: {'Header 2': 'Baselines for Object Detection'}
Content snippet: In Table 2, we present baseline experiments (given in mAP) on Mask R-CNN [12], Faster R-CNN [11], an...
